## ETL/Gold/04 - Optimizacion y estadisticas post-carga
## Liquid Clustering, OPTIMIZE, ANALYZE y (opcional) VACUUM

In [0]:
%py

def get_param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

CATALOG   = get_param("catalog", "pf")
IS_FULL   = get_param("mode", "incremental") == "full_refresh"   # solo VACUUM en full

from pyspark.sql import SparkSession

In [0]:
%py

spark.sql("ALTER TABLE pf.gold.fact_metricas_diarias CLUSTER BY (model_id, fecha_id)")



In [0]:
%py

spark.sql("OPTIMIZE pf.gold.fact_metricas_diarias")
spark.sql("OPTIMIZE pf.gold.dim_modelo_scd2 ZORDER BY (model_id, is_current)")
spark.sql("OPTIMIZE pf.silver.modelos")
spark.sql("OPTIMIZE pf.silver.model_tag")

In [0]:
%py

for t in [
    "pf.gold.fact_metricas_diarias",
    "pf.gold.dim_modelo_scd2",
    "pf.gold.dim_fecha",
    "pf.gold.dim_organizacion",
    "pf.gold.dim_task",
    "pf.gold.dim_libreria",
    "pf.gold.dim_licencia",
    "pf.gold.dim_tag",
]:
    spark.sql(f"ANALYZE TABLE {t} COMPUTE STATISTICS")

In [0]:
%py

if IS_FULL:
    spark.sql("VACUUM pf.gold.fact_metricas_diarias RETAIN 168 HOURS")
    spark.sql("VACUUM pf.bronze.models_raw RETAIN 168 HOURS")
    print("VACUUM aplicado (mantenimiento trimestral)")
else:
    print("Modo incremental: se omiten VACUUM (mantenimiento solo en full_refresh)")

In [0]:
%py

print("Optimizacion y estadisticas completadas.")